# CECS 385 — Quantum vs. Classical Portfolio Selection (20 ETFs)

**Status:** Research prototype / preliminary experiment.  
**Question:** Given identical AI-estimated returns and risk inputs, how closely does QAOA approximate the exact classical optimum when selecting **three of 20** assets?

This version uses only the adjusted-price CSV from [manisahni/marketdata](https://github.com/manisahni/marketdata), **not Yahoo Finance, Kaggle, or synthetic prices**. It compares exhaustive search, a greedy baseline, and QAOA. The January 2025 return is a single held-out observation, **not evidence that any algorithm predicts the market better**.

**Note:** This dataset is a mix of equity, bond, gold and other ETFs; call it a *multi-asset ETF universe*, not 20 equity-sector ETFs.

## 1. Install libraries
Run this once in GitHub Codespaces/VS Code or Colab, then select the notebook Python kernel if needed.

In [ ]:
%pip install -q pandas numpy scipy scikit-learn matplotlib qiskit qiskit-aer

## 2. Imports and settings
`QUICK_TEST=True` uses 12 optimizer iterations and 128 shots per evaluation. It is for checking whether the program runs, **not** for a final scientific conclusion.

In [ ]:
import time
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from scipy.optimize import minimize
from qiskit import QuantumCircuit, transpile
from qiskit_aer import AerSimulator

TICKERS = [
    "XLK", "XLV", "XLF", "XLE", "XLI", "XLP", "XLU", "XLB", "XLY", "SPY",
    "IWM", "EFA", "EEM", "TLT", "IEF", "GLD", "LQD", "VNQ", "TIP", "HYG"
]
CHOOSE = 3
RISK_LAMBDA = 1.0
QUICK_TEST = True
DEPTH = 1 if QUICK_TEST else 2
SHOTS = 128 if QUICK_TEST else 1024
MAX_ITER = 12 if QUICK_TEST else 75
SEED = 7
CIRCUIT_SCALE = 100.0
DATA_URL = "https://raw.githubusercontent.com/manisahni/marketdata/main/daily_adjclose.csv"
print("Assets:", len(TICKERS), "valid portfolios:", len(list(itertools.combinations(TICKERS, CHOOSE))))

## 3. Load the GitHub dataset
The source supplies a `Date` column and wide adjusted-close prices. The code checks for all 20 tickers, missing entries, and valid price values. **If fetching fails, it stops; it never silently generates prices.**

In [ ]:
prices_all = pd.read_csv(DATA_URL, index_col="Date", parse_dates=True).sort_index()
missing = [t for t in TICKERS if t not in prices_all.columns]
if missing:
    raise ValueError("These tickers are not in the GitHub dataset: " + ", ".join(missing))

prices = prices_all.loc["2015-01-01":"2025-01-31", TICKERS].copy()
if prices.empty or prices.isna().any().any() or (prices <= 0).any().any():
    raise ValueError("Missing or invalid price data. Check the GitHub source before continuing.")

monthly_prices = prices.resample("ME").last()
monthly_returns = monthly_prices.pct_change(fill_method=None)
print("Data source:", DATA_URL)
print("Dates:", prices.index.min().date(), "to", prices.index.max().date())
print("Daily rows:", len(prices), "| ETFs:", len(prices.columns))
display(prices.head())

## 4. Predict January 2025 returns with Random Forest
Use information available by **December 31, 2024**. Training feature dates stop at November 2024 because their labels are the following month's returns. January 2025 is held out.

The inputs are: last-month return, last three-month return, last six-month return, and recent volatility. Each fund has its own Random Forest (same settings for every fund).

In [ ]:
predictions = []
for ticker in TICKERS:
    r = monthly_returns[ticker]
    df = pd.DataFrame({
        "last_month": r,
        "last_3_months": monthly_prices[ticker].pct_change(3),
        "last_6_months": monthly_prices[ticker].pct_change(6),
        "volatility_3m": r.rolling(3).std(),
        "next_month": r.shift(-1)
    })
    columns = ["last_month", "last_3_months", "last_6_months", "volatility_3m"]
    train = df.loc[:"2024-11-30"].dropna()
    forecast = df.loc[[pd.Timestamp("2024-12-31")], columns]
    if len(train) < 24 or forecast.isna().any().any():
        raise ValueError("Not enough clean training data for " + ticker)
    model = RandomForestRegressor(n_estimators=100, min_samples_leaf=3,
                                  random_state=SEED, n_jobs=-1)
    model.fit(train[columns], train["next_month"])
    predictions.append(float(model.predict(forecast)[0]))

mu = np.array(predictions)
cov = monthly_returns.loc["2022-01-31":"2024-12-31", TICKERS].cov().to_numpy()
actual_jan = monthly_returns.loc[pd.Timestamp("2025-01-31"), TICKERS].to_numpy()
if not (np.isfinite(mu).all() and np.isfinite(cov).all() and np.isfinite(actual_jan).all()):
    raise ValueError("Non-finite values detected")
display(pd.DataFrame({"ETF": TICKERS, "Predicted Jan": mu,
                      "Actual Jan": actual_jan}).round(4))

## 5. Classical and greedy optimization
All methods optimize the same **monthly** objective:

\[\mathrm{Score}(w) = \hat\mu^T w - \lambda w^T\Sigma w\]

Each selected ETF receives 1/3 of the portfolio. Exhaustive search checks all **1,140** valid portfolios and gives the reference optimum. Greedy adds one ETF at a time.

In [ ]:
def portfolio_score(ids):
    weights = np.zeros(len(TICKERS))
    weights[list(ids)] = 1 / CHOOSE
    return float(mu @ weights - RISK_LAMBDA * (weights @ cov @ weights))

def names(ids):
    return [TICKERS[i] for i in ids]

start = time.perf_counter()
all_choices = list(itertools.combinations(range(len(TICKERS)), CHOOSE))
all_scores = np.array([portfolio_score(ids) for ids in all_choices])
best_index = int(all_scores.argmax())
exact_ids = all_choices[best_index]
exact_score = float(all_scores[best_index])
exact_seconds = time.perf_counter() - start

start = time.perf_counter()
greedy_ids = []
while len(greedy_ids) < CHOOSE:
    choices = [i for i in range(len(TICKERS)) if i not in greedy_ids]
    greedy_ids.append(max(choices, key=lambda i: portfolio_score(greedy_ids + [i])))
greedy_ids = tuple(sorted(greedy_ids))
greedy_score = portfolio_score(greedy_ids)
greedy_seconds = time.perf_counter() - start

print("Classical:", names(exact_ids), "score:", round(exact_score, 6))
print("Greedy:   ", names(greedy_ids), "score:", round(greedy_score, 6))

## 6. Build the QAOA circuit
A 20-qubit circuit uses an **XY mixer** that preserves exactly three selected assets. A phase separator encodes the same objective as the classical calculation. This is a **simulated** circuit on classical hardware.

The quick test performs a short optimization, so failure to find the optimum is not a general finding about quantum computing.

In [ ]:
N = len(TICKERS)
a = mu / CHOOSE - RISK_LAMBDA * np.diag(cov) / CHOOSE**2
b = np.zeros((N, N))
for i in range(N):
    for j in range(i + 1, N):
        b[i, j] = -2 * RISK_LAMBDA * cov[i, j] / CHOOSE**2

start_bits = [0, 7, 14]

def build_circuit(gammas, betas):
    qc = QuantumCircuit(N)
    for i in start_bits:
        qc.x(i)
    for gamma, beta in zip(gammas, betas):
        for i in range(N):
            pair_sum = sum(b[min(i, j), max(i, j)] for j in range(N) if i != j)
            qc.rz(CIRCUIT_SCALE * gamma * (a[i] + pair_sum / 2), i)
        for i in range(N):
            for j in range(i + 1, N):
                if abs(b[i, j]) > 1e-12:
                    qc.rzz(-CIRCUIT_SCALE * gamma * b[i, j] / 2, i, j)
        for offset in (0, 1):
            for i in range(offset, N - 1, 2):
                qc.rxx(beta, i, i + 1)
                qc.ryy(beta, i, i + 1)
        qc.rxx(beta, N - 1, 0)
        qc.ryy(beta, N - 1, 0)
    qc.measure_all()
    return qc

simulator = AerSimulator(method="matrix_product_state")

def sample_qaoa(theta, shots, simulator_seed):
    circuit = build_circuit(theta[:DEPTH], theta[DEPTH:])
    result = simulator.run(transpile(circuit, simulator, optimization_level=1),
                           shots=shots, seed_simulator=simulator_seed).result()
    counts = result.get_counts()
    valid = []
    for bitstring, count in counts.items():
        ids = tuple(i for i, bit in enumerate(bitstring[::-1]) if bit == "1")
        if len(ids) == CHOOSE:
            valid.append((ids, count, portfolio_score(ids)))
    return valid


## 7. Run QAOA and compare
The optimizer maximizes the **average sampled score** (by minimizing its negative). After tuning, the program selects the highest-scoring feasible portfolio observed in a larger final sample. Runtime includes optimization **and** final sampling.

For reproducibility, the simulator seed changes with every evaluation; this is still a stochastic estimate. QAOA has no guarantee of reaching the exact optimum.

In [ ]:
rng = np.random.default_rng(SEED)
calls = [0]

def loss(theta):
    calls[0] += 1
    sampled = sample_qaoa(theta, SHOTS, simulator_seed=SEED + calls[0])
    if not sampled:
        return 1e4
    count = sum(row[1] for row in sampled)
    return -sum(row[1] * row[2] for row in sampled) / count

initial = rng.uniform(0, np.pi, 2 * DEPTH)
start = time.perf_counter()
fit = minimize(loss, initial, method="COBYLA", options={"maxiter": MAX_ITER})
final = sample_qaoa(fit.x, max(1024, SHOTS), simulator_seed=SEED + 100000)
qaoa_seconds = time.perf_counter() - start
if not final:
    raise RuntimeError("No valid three-ETF sample was measured")
quantum_ids, _, quantum_score = max(final, key=lambda row: row[2])
valid_shots = sum(count for _, count, _ in final)

print("QAOA selection:", names(quantum_ids))
print("QAOA score:", round(quantum_score, 6))
print("Exact optimum:", round(exact_score, 6))
print("Gap:", round(exact_score - quantum_score, 6))
print("Found optimum:", bool(np.isclose(quantum_score, exact_score, atol=1e-9)))
print("Valid final shots:", valid_shots)
print("Optimization calls:", calls[0])
print("Full QAOA runtime:", round(qaoa_seconds, 3), "s")

## 8. Results table and visualization
January 2025 returns are computed **without trading costs**; they're only a one-month illustrative historical holdout. A reliable Sharpe-ratio comparison needs multiple matched months.

In [ ]:
results = pd.DataFrame([
    {"Method": "Classical exact", "ETFs": ", ".join(names(exact_ids)),
     "Score": exact_score, "Seconds": exact_seconds,
     "January return": float(np.mean(actual_jan[list(exact_ids)]))},
    {"Method": "Greedy", "ETFs": ", ".join(names(greedy_ids)),
     "Score": greedy_score, "Seconds": greedy_seconds,
     "January return": float(np.mean(actual_jan[list(greedy_ids)]))},
    {"Method": "Quantum QAOA", "ETFs": ", ".join(names(quantum_ids)),
     "Score": quantum_score, "Seconds": qaoa_seconds,
     "January return": float(np.mean(actual_jan[list(quantum_ids)]))}
])
results["Gap from exact"] = exact_score - results["Score"]
display(results.round(6))
results.to_csv("quantum_vs_classical_github_20_etfs.csv", index=False)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar(results["Method"], results["Gap from exact"])
axes[0].set_title("Objective gap (smaller is better)")
axes[0].set_ylabel("Exact score minus result")
axes[1].bar(results["Method"], results["Seconds"])
axes[1].set_title("Total measured runtime")
axes[1].set_ylabel("Seconds")
for ax in axes:
    ax.tick_params(axis="x", rotation=15)
plt.tight_layout()
plt.savefig("quantum_vs_classical_github_20_etfs.png", dpi=160)
plt.show()

## 9. Limits and next steps
- This is **one** historical decision (December 2024 → January 2025). Don't claim statistical significance or a Sharpe-ratio advantage.
- Extrapolating from 20 simulated qubits to 100 simulated qubits is not justified.
- The data source is a third-party static mirror of adjusted historical prices; preserve its URL and snapshot dates for reproducibility.
- The returns, risk matrix, and ETF choices are genuinely based on the GitHub CSV **if the data-loading cell succeeds**.
- For the professor's scaling question, repeat QAOA for 6, 8, 10, 12 and 20 ETFs and several seeds using the **same GitHub source and scoring rules**. Measure optimal-hit rate, objective gap, feasibility, and elapsed time.

**Data credit:** [manisahni/marketdata](https://github.com/manisahni/marketdata).